# Chapter 05 트리 알고리즘
## 05-2 교차 검증과 그리드 서치
* 이전까지는 문제를 간단히 해결하려고 테스트 세트를 사용. 하지만 테스트 세트로 일반화 성능을 올바르게 예측하려면 가능한 한 테스트 세트를 사용하지 말아야 함. 모델을 만들고 나서 마지막에 딱 한 번만 사용하는 것이 좋음. 그렇다면 max_depth 매개변수를 사용한 하이퍼파라미터 튜닝을 어떻게 할 수 있을까? 게다가 결정 트리는 테스트해 볼 매개변수가 많음
### 검증 세트
* 테스트 세트를 사용하지 않고 이를 측정하는 간단한 방법은 훈련 세트를 또 나누는 것! -> **검증 세트validation set**
> 테스트 세트와 검증 세트에 얼마나 많은 샘플을 덜어 놔야 하나요?
> * 보통 20~30%를 테스트 세트와 검증 세트로 떼어 놓음. 하지만 문제에 따라 다름. 훈련 데이터가 아주 많다면 단 몇 %만 떼어 놓아도 전체 데이터를 대표하는 데 문제가 없음.
* 훈련 세트에서 모델을 훈련하고 검증 세트로 모델을 평가 -> 테스트하고 싶은 매개변수를 바꿔가며 가장 좋은 모델 고름. 그다음 이 매개변수를 사용해 훈련 세트와 검증 세트를 합쳐 전체 훈련 데이터에서 모델을 다시 훈련. -> 마지막에 테스트 세트에서 최종 점수를 평가. -> 아마도 실전에 투입했을 때 테스트 세트의 점수와 비슷한 성능 기대 가능.

In [2]:
import pandas as pd
wine = pd.read_csv('https://bit.ly/wine_csv_data')

In [3]:
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

In [4]:
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42
)

In [5]:
# 훈련 세트 sub_input, sub_target과 검증 세트 val_input, val_target 만들기
sub_input, val_input, sub_target, val_target = train_test_split(
    train_input, train_target, test_size=0.2, random_state=42
)

In [6]:
print(sub_input.shape, val_input.shape) # 원래의 훈련 세트 5,197 -> 훈련세트 4,157, 검증 세트 1,040

(4157, 3) (1040, 3)


In [7]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)
print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


### 교차 검증
* 보통 많은 데이터를 훈련에 사용할수록 좋은 모델이 만들어짐. 그렇다고 검증 세트를 너무 조금 떼어 놓으면 검증 점수가 들쭉날쭉하고 불안정. -> **교차 검증cross validation**을 이용하면 안정적인 검증 점수를 얻고 훈련에 더 많은 데이터 사용 가능
* 교차 검증은 검증 세트를 떼어 내어 평가하는 과정을 여러 번 반복. 그다음 이 점수를 평균하여 최종 검증 검수를 얻음.
> 3-폴드 교차 검증이 뭔가요?
> * 훈련 세트를 세 부분으로 나눠서 교차 검증을 수행하는 것을 3-폴드 교차 검증이라고 함. 통칭 K-폴드 교차 검증(k-fold cross validation)이라고 하며, 훈련 세트를 몇 부분으로 나누냐에 따라 다르게 부름. k-겹 교차 검증이라고도 부름.
* 보통 5-폴드 교차 검증이나 10-폴드 교차 검증을 많이 사용. 이렇게 하면 데이터의 80~90%까지 훈련에 사용할 수 있음. 검증 세트가 줄어들지만 각 폴드에서 계산한 검증 점수를 평균하기 때문에 안정된 점수로 생각할 수 있음.

In [8]:
from sklearn.model_selection import cross_validate
scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.01465321, 0.01205754, 0.01272011, 0.01222634, 0.01198053]), 'score_time': array([0.00187159, 0.00163031, 0.00165057, 0.0016377 , 0.00195503]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


* `cross_validate()` 함수는 기본적으로 5-폴드 교차 검증을 수행. cv 매개변수에서 폴드 수를 바꿀 수도 있음.
* 교차 검증의 최종 점수는 test_score 키에 담긴 5개의 점수를 평균하여 얻을 수 있음.

In [9]:
import numpy as np
print(np.mean(scores['test_score']))

0.855300214703487


* 한 가지 주의할 점은 `cross_validate()`는 훈련 세트를 섞어 폴드를 나누지 않음. 앞서 우리는 `train_test_split()` 함수로 전체 데이터를 섞은 후 훈련 세트를 준비했기 때문에 따로 섞을 필요가 없음. 하지만 만약 교차 검증을 할 때 훈련 세트를 섞으려면 분할기splitter를 지정해야 함.
* 사이킷런의 분할기는 교차 검증에서 폴드를 어떻게 나눌지 결정해 줌. `cross_validate()` 함수는 기본적으로 회귀 모델일 경우 KFold 분할기를 사용하고 분류 모델일 경우 타깃 클래스를 골고루 나누기 위해 StratifiedKFold를 사용.

In [11]:
from sklearn.model_selection import StratifiedKFold
scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

0.855300214703487


* 만약 훈련 세트를 섞은 후 10-폴드 교차 검증을 수행하려면 다음과 같이 작성

In [12]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))

0.8574181117533719


### 하이퍼파라미터 튜닝
* 머신러닝 모델이 학습하는 파라미터를 모델 파라미터
* 모델이 학습할 수 없어서 사용자가 지정해야만 하는 파라미터를 하이퍼파라미터
* 사이킷런과 같은 머신러닝 라이브러리를 사용할 때 이런 하이퍼파라미터는 모두 클래스나 메서드의 매개변수로 표현됨.
* 하이퍼파라미터를 튜닝하는 작업
  * 먼저 라이브러리가 제공하는 기본값을 그대로 사용해 모델을 훈련
  * 그 다음 검증 세트의 점수나 교차 검증을 통해서 매개변수를 조금씩 바꿔 봄. 모델마다 적게는 1~2개에서, 많게는 5~6개의 매개변수를 제공.

> 사람의 개입 없이 하이퍼파라미터 튜닝을 자동으로 수행하는 기술을 'AutoML'이라고 부름
* 가령 결정 트리 모델에서 최적의 max_depth 값을 찾았다고 가정해 봄. 그다음 max_depth를 최적의 값으로 고정하고 min_samples_split을 바꿔가며 최적의 값을 찾음. 이렇게 한 매개변수의 최적값을 찾고 다른 매개변수의 최적값을 찾아도 될까? -> 틀렸음! 불행하게도 max_depth의 최적값은 min_samples_split 매개변수의 값이 바뀌면 함께 달라짐. 즉 이 두 매개변수를 동시에 바꿔가며 최적의 값을 찾아야 함! 게다가 매개변수가 많아지면 문제는 더 복잡해짐. -> 사이킷런에서 제공하는 **그리드 서치Grid Search**를 사용
* 사이킷런의 GridSearchCV 클래스는 친절하게도 하이퍼파라미터 탐색과 교차 검증을 한 번에 수행. 별도로 `cross_validate()` 함수를 호출할 필요가 없음.

In [14]:
from sklearn.model_selection import GridSearchCV
params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]} # 0.0001부터 0.0005까지 0.0001씩 증가하는 5개의 값 시도

In [15]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=1) # 결정 트리 클래스의 객체를 생성하자마자 바로 전달

* GridSearchCV의 cv 매개변수 기본값은 5. 따라서 min_impurity_decrease 값마다 5-폴드 교차 검증을 수행. 결국 5 $\times$ 5 = 25개의 모델을 훈련! 많은 모델을 훈련하기 때문에 GridSearchCV 클래스의 n_jobs 매개변수에서 병렬 시행에 사용할 CPU 코어 수를 지정하는 것이 좋음. 이 매개변수의 기본값은 1. -1로 지정하면 시스템에 있는 모든 코어를 사용.

In [16]:
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

* 사이킷런의 그리드 서치는 훈련이 끝나면 25개의 모델 중에서 검증 점수가 가장 높은 모델의 매개변수 조합으로 전체 훈련 세트에서 자동으로 다시 모델을 훈련. 이 모델은 gs 객체의 best_estimator_ 속성에 저장되어 있음. 이 모델을 일반 결정 트리처럼 똑같이 사용 가능.

In [17]:
dt = gs.best_estimator_
print(dt.score(train_input, train_target))

0.9615162593804117


* 그리드 서치로 찾은 최적의 매개변수는 best_params_ 속성에 저장되어 있음.

In [18]:
print(gs.best_params_) # 0.0001이 가장 좋은 값으로 선택되었음

{'min_impurity_decrease': 0.0001}


In [19]:
print(gs.cv_results_['mean_test_score'])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [20]:
best_index = np.argmax(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'][best_index])

{'min_impurity_decrease': 0.0001}


* 이 과정을 정리하면,
  * 먼저 탐색할 매개변수를 지정
  * 그다음 훈련 세트에서 그리드 서치를 수행하여 최상의 평균 검증 점수가 나오는 매개변수 조합을 찾음. 이 조합은 그리드 서치 객체에 저장됨.
  * 그리드 서치는 최상의 매개변수에서 (교차 검증에 사용한 훈련 세트가 아니라) 전체 훈련 세트를 사용해 최종 모델을 훈련. 이 모델도 그리드 서치 객체에 저장됨.
* 결정 트리에서 min_impurity_decrease는 노드를 분할하기 위한 불순도 감소 최소량을 지정. 여기에다가 max_depth로 트리의 깊이를 제한하고 min_samples_split으로 노드를 나누기 위한 최소 샘플 수도 골라 보겠음.

In [23]:
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001), # 첫 번째 매개변수 값에서 시작하여 두 번째 매개변수에 도달할 때까지 세 번째 매개변수를 계속 더한 배열을 만듦
          'max_depth': range(5, 20, 1), # 5에서 20까지 1씩 증가
          'min_samples_split': range(2, 100, 10) # 2에서 100까지 10씩 증가
          }

* 따라서 이 매개변수로 수행할 교차 검증 횟수는 9 $\times$ 15 $\times$ 10 = 1,350개. 기본 5-폴드 교차 검증을 수행하므로 만들어지는 모델의 수는 6,750개!

In [24]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(5, 20),
                         'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                         'min_samples_split': range(2, 100, 10)})

In [25]:
print(gs.best_params_) # 최상의 매개변수 조합

{'max_depth': 14, 'min_impurity_decrease': np.float64(0.0004), 'min_samples_split': 12}


In [26]:
print(np.max(gs.cv_results_['mean_test_score'])) # 최상의 교차 검증 점수

0.8683865773302731


* GridSearchCV 클래스를 사용하니 매개변수를 일일이 바꿔가며 교차 검증을 수행하지 않고 원하는 매개변수 값을 나열하면 자동으로 교차 검증을 수행해서 최상의 매개변수를 찾을 수 있음.
* 앞에서 탐색할 매개변수의 간격을 0.0001 혹은 1로 설정했는데, 이렇게 간격을 둔 것에 특별한 근거가 없음. 이보다 더 좁거나 넓은 간격으로 시도해 볼 수 있지 않을까?

### 랜덤 서치
* 매개변수의 값이 수치일 때 값의 범위나 간격을 미리 정하기 어려울 수 있음. 또 너무 많은 매개변수 조건이 있어 그리드 서치 수행 시간이 오래 걸릴 수 있음. 이럴 때 **랜덤 서치Random search**를 사용하면 좋음.
* 랜덤 서치에는 매개변수 값의 목록을 전달하는 것이 아니라 매개변수를 샘플링할 수 있는 확률 분포 객체를 전달함.
> 싸이파이(scipy)는 어떤 라이브러리인가요?
> * 싸이파이는 파이썬의 핵심 과학 라이브러리 중 하나. 적분, 보간, 선형 대수, 확률 등을 포함한 수치 계산 전용 라이브러리. 사이킷런은 넘파이와 싸이파이 기능을 많이 사용.

In [27]:
from scipy.stats import uniform, randint

* 싸이파이의 stats 서브 패키지에 있는 uniform과 randint 클래스는 모두 주어진 범위에서 고르게 값을 뽑음. 이를 '균등 분포에서 샘플링한다'고 말함. randint는 정숫값을 뽑고, uniform은 실숫값을 뽑음.

In [28]:
rgen = randint(0, 10)
rgen.rvs(10)

array([3, 1, 1, 2, 4, 5, 0, 0, 4, 8])

In [29]:
np.unique(rgen.rvs(1000), return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([ 99, 119,  96, 103,  89,  92, 112,  96,  99,  95]))

In [30]:
ugen = uniform(0, 1)
ugen.rvs(10)

array([0.39858381, 0.72102876, 0.43864659, 0.28053153, 0.9463143 ,
       0.38363142, 0.54670875, 0.33186151, 0.63164235, 0.97344867])

* 랜덤 서치에 randint과 uniform 클래스 객체를 넘겨주고 총 몇 번을 샘플링해서 최적의 매개변수를 찾으라고 명령할 수 있음.

* 여기서는 min_samples_leaf 매개변수를 탐색 대상에 추가. 이 매개변수는 리프 노드가 되기 위한 최소 샘플의 개수. 어떤 노드가 분할하여 만들어질 자식 노드의 샘플 수가 이 값보다 작을 경우 분할하지 않음.

In [31]:
params = {'min_impurity_decrease': uniform(0.0001, 0.001),
          'max_depth': randint(20, 50),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25),
          }

In [32]:
from sklearn.model_selection import RandomizedSearchCV
gs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params, n_iter=100, n_jobs=-1, random_state=42)
gs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7bb9b02fae90>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7bb9b02fb110>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7bb9b02fb610>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7bb9b02fb4d0>},
                   random_state=42)

In [33]:
print(gs.best_params_)

{'max_depth': 39, 'min_impurity_decrease': np.float64(0.00034102546602601173), 'min_samples_leaf': 7, 'min_samples_split': 13}


In [34]:
print(np.max(gs.cv_results_['mean_test_score']))

0.8695428296438884


In [35]:
dt = gs.best_estimator_
print(dt.score(test_input, test_target))

0.86


* 테스트 세트 점수는 검증 세트에 대한 점수보다 조금 작은 것이 일반적.